# Paper-based MGAT Resume Classification + Specific Job Role Prediction

Google Colab project based on *A Domain Adaptation Approach for Resume Classification Using Graph Attention Networks and Natural Language Processing*. It preserves the paper-inspired NLP + GloVe + co-occurrence graph + multi-head GAT pipeline for broad occupational groups and adds a **specific-role prediction extension** (e.g., DevOps Engineer, Backend Engineer) trained from job-posting titles. Public substitute datasets are used because the paper's original proprietary datasets are not included.


## 1. Runtime and experiment design

In Colab choose **Runtime → Change runtime type → T4 GPU** if available, then run all cells in order. No local files or manual dataset uploads are needed.

**How this follows the paper**
- Source domain: labelled job postings.
- Target domain: resume text; its category labels are used only for final evaluation.
- Text processing: spaCy POS filtering, stop-word removal and lemmatization.
- Word vectors: 300-dimensional GloVe.
- Graph: word co-occurrence with context window 3 and corpus-level edge weights.
- Model: two GAT layers with 8 attention heads, graph-level mean pooling and a classifier.
- Evaluation: accuracy, macro-F1, macro-AUC where possible, per-class report, confusion matrix, and TF-IDF + LinearSVC baseline.

**Improvements over the earlier notebook:** automatic data loading; no target labels passed into training batches; shared corpus co-occurrence weights; source-title label leakage reduced by excluding the title from model input; stronger per-class evaluation and baseline comparison.

**Role-model fix:** the specific-role model now trains from an independent, larger job-posting sample rather than the already filtered broad-category sample. It removes duplicate feature text, uses word + character TF-IDF features, requires at least 25 examples per role, and reports honest held-out metrics. The full notebook has not been executed end-to-end in this environment.


In [ ]:
%pip install -q "datasets>=2.18" huggingface_hub pypdf pandas numpy scikit-learn matplotlib seaborn "gensim>=4.4.0" spacy torch-geometric tqdm
!python -m spacy download en_core_web_sm

In [ ]:
import os, re, json, random, collections, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from tqdm.auto import tqdm
import spacy, gensim.downloader as api

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 2. Load public datasets

The resume dataset contains 2,484 labelled resumes. The public job-postings dataset contains over one million records, so we **stream and sample** it instead of downloading the entire CSV.

The public job dataset does not include the paper's manually curated 10 labels. We infer a broad group from each job's title using explicit keyword rules. This is weak supervision and can introduce label noise.

In [ ]:
REPO="mithinsagar/exai-resumeintel-data"
resume_df=load_dataset(REPO, "resumes", split="train").to_pandas()
print("Resume columns:", resume_df.columns.tolist(), "rows:", len(resume_df))
display(resume_df.head(3))

MAX_SOURCE_PER_CLASS=250
MAX_SCAN=150000
job_stream=load_dataset(REPO, "job_postings", split="train", streaming=True).shuffle(seed=SEED, buffer_size=50000)
print("Job-posting stream loaded and shuffled for broader category coverage.")

## 3. Map source and target categories
Resume categories are mapped into paper-style broad groups. Job labels are weakly inferred from job titles. Unmapped categories are excluded rather than guessed.

In [ ]:
PAPER_GROUPS=["IT software","Industrial management","Technical","Medical","Finance and banking","Back-office","Sales","Media and marketing","Field","Hospitality"]

RESUME_MAP={
"INFORMATION-TECHNOLOGY":"IT software",
"BUSINESS-DEVELOPMENT":"Sales",
"FINANCE":"Finance and banking","ACCOUNTANT":"Finance and banking","BANKING":"Finance and banking",
"ENGINEERING":"Technical","AUTOMOBILE":"Technical",
"HEALTHCARE":"Medical",
"SALES":"Sales",
"CONSTRUCTION":"Field",
"PUBLIC-RELATIONS":"Media and marketing","DIGITAL-MEDIA":"Media and marketing",
"HR":"Back-office","BPO":"Back-office",
"CHEF":"Hospitality","AVIATION":"Hospitality",
"APPAREL":"Industrial management"
}

# Ordered rules: more specific categories are checked before generic Technical.
JOB_RULES=[
("Medical", r"\b(nurse|nursing|doctor|physician|medical|healthcare|clinical|pharmacist|dentist|therapist|surgeon|hospital|radiolog|paramedic|veterinar)\b"),
("Finance and banking", r"\b(finance|financial|accountant|accounting|banker|banking|bank|tax|auditor|audit|investment|payroll|actuarial|treasury|credit analyst|risk analyst)\b"),
("IT software", r"\b(software|developer|programmer|web developer|frontend|front-end|backend|back-end|full stack|data scientist|data engineer|machine learning|artificial intelligence|cybersecurity|network administrator|cloud engineer|devops|database administrator|it support|systems analyst|soc analyst|security operations|automation tester)\b"),
("Hospitality", r"\b(chef|cook|hotel|hospitality|airline|aviation|flight attendant|travel agent|tourism|tour guide|reservation agent|catering)\b"),
("Field", r"\b(construction|civil engineer|site engineer|architect|surveyor|agricultur|farm manager|landscape architect|building inspector|quantity surveyor)\b"),
("Media and marketing", r"\b(marketing|digital media|social media|content writer|public relations|journalist|advertising|brand manager|graphic designer|ui.?ux designer|art director|copywriter|seo specialist)\b"),
("Sales", r"\b(sales|business development|account executive|retail associate|sales representative|sales manager|territory manager|customer acquisition)\b"),
("Industrial management", r"\b(supply chain|logistics|warehouse|manufacturing|production manager|procurement|purchasing|quality control|industrial engineer|operations manager|inventory manager|plant manager)\b"),
("Back-office", r"\b(human resources|\bhr\b|recruiter|recruitment|administrator|administrative|office manager|data entry|record keeper|records clerk|bpo|customer support|customer service representative|executive assistant)\b"),
("Technical", r"\b(engineer|engineering|technician|mechanic|automotive|electrical|mechanical|electronics|hardware|maintenance technician|instrumentation)\b")
]
def infer_job_group(role):
    s=str(role).lower()
    for group, pattern in JOB_RULES:
        if re.search(pattern,s): return group
    return None

cat_col=next((c for c in resume_df.columns if c.lower() in ["category","label","occupation"]),None)
text_col=next((c for c in resume_df.columns if c.lower() in ["feature","resume","resume_text","text","content"]),None)
if cat_col is None or text_col is None:
    raise ValueError("Resume dataset schema changed. Found columns: "+str(resume_df.columns.tolist()))
resume_df=resume_df[[cat_col,text_col]].rename(columns={cat_col:"raw_category",text_col:"text"})
resume_df["text"]=resume_df["text"].fillna("").astype(str)
resume_df["group"]=resume_df["raw_category"].astype(str).str.upper().map(RESUME_MAP)
print("Resume categories not mapped to paper groups (excluded):",
      sorted(resume_df.loc[resume_df["group"].isna(),"raw_category"].astype(str).unique().tolist()))
resume_df=resume_df[resume_df["group"].notna() & (resume_df["text"].str.len()>30)].copy()

job_rows=[]; counts=collections.Counter()
for i,row in enumerate(job_stream):
    role=row.get("Role",row.get("role",""))
    features=row.get("Features",row.get("features",""))
    group=infer_job_group(role)
    if group and counts[group]<MAX_SOURCE_PER_CLASS:
        # Infer the label from Role, but do NOT include Role in model input.
        # This reduces the shortcut/leakage where the label words appear directly in the text.
        txt=str(features).strip()
        if len(txt)>30 and txt.lower() not in {"nan", "none"}:
            job_rows.append({"role":str(role),"text":txt,"group":group})
            counts[group]+=1
    if i+1>=MAX_SCAN or all(counts[g]>=MAX_SOURCE_PER_CLASS for g in PAPER_GROUPS):
        break

job_df=pd.DataFrame(job_rows)
print("Scanned:",i+1,"job records; retained:",len(job_df))
print("Job groups:"); display(job_df["group"].value_counts().to_frame("job_count"))
print("Resume groups (used only for evaluation):"); display(resume_df["group"].value_counts().to_frame("resume_count"))
jc=job_df["group"].value_counts(); rc=resume_df["group"].value_counts()
# Determine the shared label space from source support and the pre-declared resume taxonomy.
# Do not use target label frequencies to choose classes; target labels remain evaluation-only.
resume_taxonomy_groups=set(RESUME_MAP.values())
SHARED_GROUPS=sorted([g for g in PAPER_GROUPS if jc.get(g,0)>=30 and g in resume_taxonomy_groups])
print("Shared categories:",SHARED_GROUPS)
if len(SHARED_GROUPS)<3:
    raise RuntimeError("Fewer than 3 source categories found. Increase MAX_SCAN to 250000 and rerun this cell.")
job_df=job_df[job_df.group.isin(SHARED_GROUPS)].reset_index(drop=True)
resume_df=resume_df[resume_df.group.isin(SHARED_GROUPS)].reset_index(drop=True)
display(pd.DataFrame({"jobs":job_df.group.value_counts(),"resumes":resume_df.group.value_counts()}).fillna(0).astype(int))

## 3. Preprocess text, load GloVe, and build shared co-occurrence weights

The paper builds a shared word representation from job-post and resume content. Here, we construct corpus-level word-pair counts from **text only** (no target labels), using a context window of 3. Each document is then represented as a word graph with GloVe node vectors and normalized corpus-level edge weights.

For Colab memory, each document graph is capped at 120 unique words. This is a practical limit, not a parameter confirmed to match the original paper.

In [ ]:
nlp=spacy.load("en_core_web_sm",disable=["ner","parser"])
print("Loading GloVe 300d vectors; first run can take a few minutes.")
glove=api.load("glove-wiki-gigaword-300")

WINDOW_SIZE=3
MAX_NODES=120
TOKEN_CACHE={}

def doc_to_tokens(text):
    key=str(text)
    if key in TOKEN_CACHE:
        return TOKEN_CACHE[key]
    doc=nlp(key[:30000])
    out=[]
    for tok in doc:
        w=tok.lemma_.lower().strip()
        if tok.is_alpha and len(w)>1 and tok.pos_ in {"NOUN","ADJ","VERB","ADV"} and w not in nlp.Defaults.stop_words and w in glove:
            out.append(w)
    TOKEN_CACHE[key]=out
    return out

# Unsupervised shared corpus co-occurrence: source and target TEXT are allowed;
# target category labels are never used here.
GLOBAL_COOCCURRENCE=collections.Counter()
all_texts=list(job_df["text"].astype(str))+list(resume_df["text"].astype(str))
for text in tqdm(all_texts,desc="Building shared word co-occurrence"):
    seq=doc_to_tokens(text)
    for i,w1 in enumerate(seq):
        for j in range(i+1,min(i+WINDOW_SIZE+1,len(seq))):
            w2=seq[j]
            if w1 != w2:
                GLOBAL_COOCCURRENCE[tuple(sorted((w1,w2)))]+=1
MAX_LOG_WEIGHT=max([np.log1p(v) for v in GLOBAL_COOCCURRENCE.values()],default=1.0)
print("Cached documents:",len(TOKEN_CACHE),"unique word pairs:",len(GLOBAL_COOCCURRENCE))

def make_graph(text,label_id=None):
    seq=doc_to_tokens(text)
    is_empty=(len(seq)==0)
    words=["__empty__"] if is_empty else list(dict.fromkeys(seq))[:MAX_NODES]
    idx={w:i for i,w in enumerate(words)}
    if is_empty:
        x=np.zeros((1,300),dtype=np.float32)
    else:
        x=np.stack([glove[w] for w in words]).astype(np.float32)

    edge_weights={}
    for i,w1 in enumerate(seq):
        if w1 not in idx: continue
        for j in range(i+1,min(i+WINDOW_SIZE+1,len(seq))):
            w2=seq[j]
            if w2 in idx and idx[w1]!=idx[w2]:
                a,b=idx[w1],idx[w2]
                pair=tuple(sorted((w1,w2)))
                raw=GLOBAL_COOCCURRENCE.get(pair,1)
                weight=float(np.log1p(raw)/MAX_LOG_WEIGHT) if MAX_LOG_WEIGHT else 1.0
                edge_weights[(a,b)]=weight
                edge_weights[(b,a)]=weight

    if edge_weights:
        edge_index=torch.tensor(list(edge_weights.keys()),dtype=torch.long).t().contiguous()
        edge_attr=torch.tensor([[v] for v in edge_weights.values()],dtype=torch.float32)
    else:
        edge_index=torch.empty((2,0),dtype=torch.long)
        edge_attr=torch.empty((0,1),dtype=torch.float32)

    g=Data(x=torch.tensor(x,dtype=torch.float32),edge_index=edge_index,edge_attr=edge_attr)
    if label_id is not None:
        g.y=torch.tensor([label_id],dtype=torch.long)
    return g

label_to_id={g:i for i,g in enumerate(SHARED_GROUPS)}
id_to_label={i:g for g,i in label_to_id.items()}
print("Labels:",label_to_id)

## 4. Source-only training/validation split

Only source job-post labels are used to train and select the checkpoint. Target resume graphs are created **without labels attached**. Resume categories are held separately and will be compared with predictions only after training.

In [ ]:
job_df=job_df[job_df.group.isin(SHARED_GROUPS)].reset_index(drop=True)
resume_df=resume_df[resume_df.group.isin(SHARED_GROUPS)].reset_index(drop=True)
job_train_df,job_val_df=train_test_split(job_df,test_size=0.20,random_state=SEED,stratify=job_df.group)

def build_source_graphs(frame,desc):
    graphs=[]
    for row in tqdm(frame.itertuples(index=False),total=len(frame),desc=desc):
        graphs.append(make_graph(row.text,label_to_id[row.group]))
    return graphs

train_graphs=build_source_graphs(job_train_df,"Source train graphs")
val_graphs=build_source_graphs(job_val_df,"Source validation graphs")
target_graphs=[make_graph(text) for text in tqdm(resume_df.text.astype(str),desc="Unlabelled target resume graphs")]

BATCH_SIZE=32
train_loader=DataLoader(train_graphs,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_graphs,batch_size=BATCH_SIZE,shuffle=False)
target_loader=DataLoader(target_graphs,batch_size=BATCH_SIZE,shuffle=False)
print("Graph counts:",len(train_graphs),len(val_graphs),len(target_graphs))
print("Target labels are stored separately for evaluation only.")

## 5. Two-layer multi-head Graph Attention Network

The model uses 8 attention heads and passes corpus-level co-occurrence weights as edge features. This is closer to the paper's weighted word co-occurrence graph than a plain unweighted per-document graph.

In [ ]:
class MGATClassifier(nn.Module):
    def __init__(self,n_classes,hidden=64,heads=8,dropout=0.3):
        super().__init__()
        self.dropout=dropout
        self.gat1=GATConv(300,hidden,heads=heads,concat=True,dropout=dropout,edge_dim=1)
        self.gat2=GATConv(hidden*heads,hidden,heads=heads,concat=True,dropout=dropout,edge_dim=1)
        self.fc=nn.Sequential(nn.Linear(hidden*heads,128),nn.ReLU(),nn.Dropout(dropout),nn.Linear(128,n_classes))
    def forward(self,data):
        x,edge_index,edge_attr,batch=data.x,data.edge_index,data.edge_attr,data.batch
        x=F.dropout(x,p=self.dropout,training=self.training)
        x=F.elu(self.gat1(x,edge_index,edge_attr))
        x=F.dropout(x,p=self.dropout,training=self.training)
        x=F.elu(self.gat2(x,edge_index,edge_attr))
        return self.fc(global_mean_pool(x,batch))

model=MGATClassifier(len(SHARED_GROUPS)).to(DEVICE)
optimizer=torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=1e-4)
criterion=nn.CrossEntropyLoss()
print(model)

## 7. Train with source validation only

In [ ]:
def run_epoch(loader,training=False):
    model.train(training); loss_sum=0; n=0; yt=[]; yp=[]
    for batch in loader:
        batch=batch.to(DEVICE)
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits=model(batch); y=batch.y.view(-1); loss=criterion(logits,y)
            if training:
                loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        loss_sum+=loss.item()*len(y); n+=len(y)
        yt.extend(y.detach().cpu().tolist()); yp.extend(logits.argmax(1).detach().cpu().tolist())
    return {"loss":loss_sum/max(n,1),"accuracy":accuracy_score(yt,yp),"macro_f1":f1_score(yt,yp,average="macro",zero_division=0)}

os.makedirs("artifacts",exist_ok=True)
history=[]; best=-1
for epoch in range(1,11):
    tr=run_epoch(train_loader,True); va=run_epoch(val_loader,False)
    rec={"epoch":epoch,"train_loss":tr["loss"],"train_accuracy":tr["accuracy"],"train_macro_f1":tr["macro_f1"],
         "val_loss":va["loss"],"val_accuracy":va["accuracy"],"val_macro_f1":va["macro_f1"]}
    history.append(rec); print(rec)
    if va["macro_f1"]>best:
        best=va["macro_f1"]
        torch.save({"state":model.state_dict(),"labels":SHARED_GROUPS},"artifacts/mgat_source_only.pt")
pd.DataFrame(history).to_csv("artifacts/training_history.csv",index=False)
display(pd.DataFrame(history))

## 7. Evaluate on held-out resumes

Target labels are added only after predictions are produced. Metrics include accuracy, macro-F1, weighted-F1, macro precision/recall, balanced accuracy and macro one-vs-rest AUC where every shared class is represented.

In [ ]:
ckpt=torch.load("artifacts/mgat_source_only.pt",map_location=DEVICE,weights_only=False)
model.load_state_dict(ckpt["state"])
model.eval()

def predict_loader(loader):
    yp=[]; probs=[]
    with torch.no_grad():
        for batch in loader:
            batch=batch.to(DEVICE)
            logits=model(batch)
            p=torch.softmax(logits,dim=1)
            yp.extend(p.argmax(1).cpu().tolist())
            probs.extend(p.cpu().numpy())
    return np.array(yp),np.array(probs)

# Target labels are accessed only now, after model training and prediction.
y_true=np.array([label_to_id[x] for x in resume_df.group])
y_pred,y_prob=predict_loader(target_loader)

metrics={
    "accuracy":float(accuracy_score(y_true,y_pred)),
    "macro_f1":float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
    "weighted_f1":float(f1_score(y_true,y_pred,average="weighted",zero_division=0)),
    "macro_precision":float(precision_score(y_true,y_pred,average="macro",zero_division=0)),
    "macro_recall":float(recall_score(y_true,y_pred,average="macro",zero_division=0)),
    "balanced_accuracy":float(balanced_accuracy_score(y_true,y_pred))
}
if len(np.unique(y_true))==len(SHARED_GROUPS):
    try:
        metrics["macro_ovr_auc"]=float(roc_auc_score(y_true,y_prob,labels=np.arange(len(SHARED_GROUPS)),multi_class="ovr",average="macro"))
    except Exception as e:
        metrics["macro_ovr_auc_error"]=str(e)
else:
    metrics["macro_ovr_auc"]="Not computed: at least one shared class has no target examples."

print(json.dumps(metrics,indent=2))
print(classification_report(y_true,y_pred,labels=list(range(len(SHARED_GROUPS))),target_names=SHARED_GROUPS,zero_division=0))
with open("artifacts/mgat_target_metrics.json","w") as f: json.dump(metrics,f,indent=2)

cm=confusion_matrix(y_true,y_pred,labels=list(range(len(SHARED_GROUPS))))
plt.figure(figsize=(max(8,len(SHARED_GROUPS)),max(6,len(SHARED_GROUPS)*0.7)))
sns.heatmap(cm,annot=True,fmt="d",xticklabels=SHARED_GROUPS,yticklabels=SHARED_GROUPS,cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("MGAT cross-domain resume confusion matrix")
plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.show()

preds=resume_df[["raw_category","group","text"]].copy()
preds["predicted_group"]=[id_to_label[i] for i in y_pred]
preds["confidence"]=y_prob.max(axis=1)
preds.to_csv("artifacts/mgat_resume_predictions.csv",index=False)

## 8. Compare against TF-IDF + LinearSVC

This is a non-graph baseline trained on the same source training set and evaluated on the same target resumes. This helps show whether the MGAT actually improves cross-domain transfer instead of assuming that a graph model is better.

In [ ]:
tfidf=TfidfVectorizer(max_features=40000,ngram_range=(1,2),sublinear_tf=True,min_df=2,max_df=0.98)
X_train=tfidf.fit_transform(job_train_df.text.astype(str))
X_target=tfidf.transform(resume_df.text.astype(str))
y_source=np.array([label_to_id[x] for x in job_train_df.group])
y_target=np.array([label_to_id[x] for x in resume_df.group])
baseline=LinearSVC(class_weight="balanced",random_state=SEED)
baseline.fit(X_train,y_source); base_pred=baseline.predict(X_target)
base_metrics={"accuracy":float(accuracy_score(y_target,base_pred)),"macro_f1":float(f1_score(y_target,base_pred,average="macro",zero_division=0)),
"weighted_f1":float(f1_score(y_target,base_pred,average="weighted",zero_division=0)),
"macro_precision":float(precision_score(y_target,base_pred,average="macro",zero_division=0)),
"macro_recall":float(recall_score(y_target,base_pred,average="macro",zero_division=0)),
"balanced_accuracy":float(balanced_accuracy_score(y_target,base_pred))}
comparison=pd.DataFrame([{"model":"MGAT",**metrics},{"model":"TF-IDF + LinearSVC",**base_metrics}]).set_index("model")
display(comparison); comparison.to_csv("artifacts/model_comparison.csv")
print(classification_report(y_target,base_pred,labels=list(range(len(SHARED_GROUPS))),target_names=SHARED_GROUPS,zero_division=0))

## 10. Upload a resume and predict specific job roles

This cell independently streams a larger sample of public job postings for the role-level extension (rather than reusing the broad-category-filtered sample). It deduplicates feature text, requires minimum examples per role, combines word and character n-gram features, and reports held-out validation metrics to help detect misleadingly perfect scores. It then lets you upload a PDF and displays the top three specific roles and broad MGAT groups with model scores and percentage formatting.

The role-level extension is an improvement to the paper-inspired project, not a result reported in the paper. Job-title-derived labels are weak supervision. Percentage values are model estimates and must not be interpreted as guaranteed candidate suitability or calibrated correctness probabilities.


In [ ]:
from google.colab import files
from pypdf import PdfReader
import io, joblib, json, re, collections
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, accuracy_score

# Specific-role extension. The original paper predicts broad occupational groups;
# this extension predicts more specific roles using public job-title labels.
# Important: role titles are used to create labels, NOT as model input.
ROLE_RULES = [
    ("DevOps Engineer", r"\b(devops|site reliability engineer|site reliability|sre|platform engineer|infrastructure automation)\b"),
    ("Cloud Engineer", r"\b(cloud engineer|cloud architect|aws engineer|azure engineer|gcp engineer|cloud infrastructure)\b"),
    ("Backend Engineer", r"\b(back.?end|server.?side|api developer|java developer|python developer|node\.?js developer|software engineer backend)\b"),
    ("Frontend Engineer", r"\b(front.?end|react developer|angular developer|vue developer|ui developer)\b"),
    ("Full-Stack Developer", r"\b(full.?stack)\b"),
    ("Software Engineer", r"\b(software engineer|software developer|application developer|programmer)\b"),
    ("Data Engineer", r"\b(data engineer|etl developer|data platform engineer)\b"),
    ("Data Scientist", r"\b(data scientist|research scientist|quantitative scientist)\b"),
    ("Machine Learning Engineer", r"\b(machine learning engineer|ml engineer|ai engineer|deep learning engineer)\b"),
    ("Data Analyst", r"\b(data analyst|business intelligence analyst|bi analyst|reporting analyst)\b"),
    ("Cybersecurity Analyst", r"\b(cyber.?security|security analyst|soc analyst|penetration tester|information security)\b"),
    ("Network Engineer", r"\b(network engineer|network administrator|systems network)\b"),
    ("QA Automation Engineer", r"\b(qa automation|automation test|test automation|sdet|quality assurance automation)\b"),
    ("QA Engineer", r"\b(qa engineer|quality assurance|software tester|manual tester)\b"),
    ("Database Administrator", r"\b(database administrator|dba|database engineer)\b"),
    ("Systems Administrator", r"\b(system administrator|systems administrator|linux administrator|windows administrator)\b"),
    ("Product Manager", r"\b(product manager|technical product manager)\b"),
    ("Project Manager", r"\b(project manager|program manager)\b"),
    ("UI/UX Designer", r"\b(ui.?ux|ux designer|user experience designer|user interface designer)\b"),
    ("HR / Recruiter", r"\b(recruiter|talent acquisition|human resources|hr manager)\b"),
    ("Accountant", r"\b(accountant|accounting|tax accountant|auditor)\b"),
    ("Financial Analyst", r"\b(financial analyst|finance analyst|investment analyst|credit analyst)\b"),
    ("Sales Representative", r"\b(sales representative|sales executive|account executive|business development)\b"),
    ("Marketing Specialist", r"\b(marketing specialist|digital marketing|seo specialist|content marketing)\b"),
    ("Mechanical Engineer", r"\b(mechanical engineer|mechanical design)\b"),
    ("Electrical Engineer", r"\b(electrical engineer|electronics engineer)\b"),
    ("Civil Engineer", r"\b(civil engineer|site engineer|construction engineer)\b"),
    ("Technician", r"\b(technician|maintenance engineer|field service technician)\b"),
]

def infer_specific_role(title):
    title = re.sub(r"[^a-z0-9+#./ -]", " ", str(title).lower())
    for role, pattern in ROLE_RULES:
        if re.search(pattern, title):
            return role
    return None

# IMPORTANT IMPROVEMENT:
# The old role model reused job_df after it had already been filtered to broad
# paper categories and capped per broad class. That can leave a tiny, biased
# role dataset and produce misleadingly perfect validation scores.
# Re-stream the public postings for role-level training independently.
ROLE_SCAN_LIMIT = 300_000
MIN_ROLE_EXAMPLES = 25
ROLE_MAX_PER_CLASS = 1500
role_stream = load_dataset(REPO, "job_postings", split="train", streaming=True).shuffle(
    seed=SEED + 7, buffer_size=50_000
)
role_rows = []
role_counts = collections.Counter()
scanned = 0

for row in role_stream:
    scanned += 1
    title = row.get("Role", row.get("role", ""))
    features = row.get("Features", row.get("features", ""))
    role = infer_specific_role(title)
    text = str(features).strip()
    if role and len(text) >= 40 and text.lower() not in {"nan", "none"}:
        # Avoid including the title in the text input; this reduces direct
        # label leakage and makes inference more comparable to an uploaded resume.
        if role_counts[role] < ROLE_MAX_PER_CLASS:
            role_rows.append({"text": text, "specific_role": role})
            role_counts[role] += 1
    if scanned >= ROLE_SCAN_LIMIT:
        break

role_source = pd.DataFrame(role_rows)
if role_source.empty:
    raise RuntimeError("No specific-role examples found. Check the public dataset schema or increase ROLE_SCAN_LIMIT.")

# Remove exact duplicate feature descriptions before splitting to reduce leakage.
role_source["text"] = role_source["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
role_source = role_source.drop_duplicates(subset=["text", "specific_role"]).reset_index(drop=True)
role_counts = role_source["specific_role"].value_counts()
supported_roles = role_counts[role_counts >= MIN_ROLE_EXAMPLES].index
role_source = role_source[role_source["specific_role"].isin(supported_roles)].reset_index(drop=True)

print(f"Scanned {scanned:,} postings; retained {len(role_source):,} deduplicated role-labelled examples.")
print("Training examples per specific role:")
display(role_source["specific_role"].value_counts().to_frame("examples"))

if role_source["specific_role"].nunique() < 2:
    raise RuntimeError(
        "Fewer than two job roles have enough examples. Increase ROLE_SCAN_LIMIT or lower MIN_ROLE_EXAMPLES modestly."
    )

X_train_role, X_val_role, y_train_role, y_val_role = train_test_split(
    role_source["text"], role_source["specific_role"],
    test_size=0.20, random_state=SEED,
    stratify=role_source["specific_role"]
)

# Word n-grams capture technical terms; character n-grams help with variants
# such as node.js/NodeJS, Kubernetes/K8s, CI/CD and spelling variants.
role_features = FeatureUnion([
    ("word", TfidfVectorizer(
        max_features=60000, ngram_range=(1, 2), min_df=2,
        sublinear_tf=True, strip_accents="unicode", stop_words="english"
    )),
    ("char", TfidfVectorizer(
        analyzer="char_wb", max_features=40000, ngram_range=(3, 5),
        min_df=2, sublinear_tf=True
    )),
])
role_model = Pipeline([
    ("features", role_features),
    ("classifier", LogisticRegression(
        max_iter=2000, class_weight="balanced", C=2.0,
        solver="lbfgs", random_state=SEED
    ))
])
role_model.fit(X_train_role, y_train_role)
role_val_pred = role_model.predict(X_val_role)
role_val_f1 = f1_score(y_val_role, role_val_pred, average="macro", zero_division=0)
role_val_acc = accuracy_score(y_val_role, role_val_pred)
print(f"Role-level held-out validation accuracy: {role_val_acc:.4f}")
print(f"Role-level held-out validation macro-F1: {role_val_f1:.4f}")
print("Held-out report (these scores measure title-derived labels, not the paper's MGAT result):")
print(classification_report(y_val_role, role_val_pred, zero_division=0))

os.makedirs("artifacts", exist_ok=True)
joblib.dump(role_model, "artifacts/specific_role_model.joblib")
with open("artifacts/specific_role_labels.json", "w") as f:
    json.dump({
        "roles": sorted(role_model.classes_.tolist()),
        "min_role_examples": MIN_ROLE_EXAMPLES,
        "training_examples": int(len(role_source)),
        "validation_accuracy": float(role_val_acc),
        "validation_macro_f1": float(role_val_f1),
        "label_source": "job-posting titles; weak supervision"
    }, f, indent=2)

def predict_resume(text):
    """Predict specific roles and paper-style broad MGAT groups for resume text."""
    if not isinstance(text, str) or len(text.strip()) < 30:
        raise ValueError("Resume text is empty or too short. Upload a text-based PDF.")
    role_probs = role_model.predict_proba([text])[0]
    role_order = np.argsort(role_probs)[::-1][:3]
    # Show probabilities as percentages, but note these are model estimates and
    # not guaranteed to be calibrated measures of candidate suitability.
    specific = pd.DataFrame([
        {
            "rank": rank + 1,
            "predicted_job_role": str(role_model.classes_[j]),
            "model_score": float(role_probs[j]),
            "percentage": f"{100.0 * float(role_probs[j]):.2f}%"
        }
        for rank, j in enumerate(role_order)
    ])

    graph = make_graph(text, None)
    batch = Batch.from_data_list([graph]).to(DEVICE)
    model.eval()
    with torch.no_grad():
        group_probs = torch.softmax(model(batch), dim=1)[0].cpu().numpy()
    group_order = np.argsort(group_probs)[::-1][:3]
    broad = pd.DataFrame([
        {
            "rank": rank + 1,
            "predicted_category": id_to_label[int(j)],
            "model_score": float(group_probs[j]),
            "percentage": f"{100.0 * float(group_probs[j]):.2f}%"
        }
        for rank, j in enumerate(group_order)
    ])
    return specific, broad

def extract_pdf_text(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    return "\n".join(page.extract_text() or "" for page in reader.pages).strip()

print("Upload a candidate resume PDF. Output shows specific roles plus paper-style broad categories.")
uploaded = files.upload()
for filename, data in uploaded.items():
    if not filename.lower().endswith(".pdf"):
        print(f"Skipping {filename}: please upload a PDF file.")
        continue
    resume_text = extract_pdf_text(data)
    if len(resume_text) < 30:
        print(f"Could not extract enough text from {filename}. Scanned PDFs require OCR.")
        continue
    print(f"\nResume: {filename}")
    specific_roles, broad_groups = predict_resume(resume_text)
    print("Top specific job roles (role-level extension):")
    display(specific_roles)
    print("Top broad occupational groups (paper-style MGAT):")
    display(broad_groups)


## 11. Download results

In [ ]:
import shutil
shutil.make_archive("mgat_results","zip","artifacts")
from google.colab import files
files.download("mgat_results.zip")

## Limitations, reproducibility, and responsible use

- **Original data unavailable:** the paper used 45,200 job postings and 46,230 resumes from a private headhunter source. This project uses public substitute data, so it is not an exact reproduction.
- **Paper-faithful component:** the MGAT pipeline predicts broad occupational groups from source job-posting labels and evaluates on resume data without using resume labels during training.
- **Specific-role extension:** the added role model learns specific role labels inferred from job-posting titles. These labels are weak supervision and depend on source-data quality; the role-level extension is not the paper's reported MGAT result.
- **Improvement:** the role model independently samples job postings, removes duplicate feature text, requires minimum class support, uses word and character n-grams, and reports held-out validation accuracy and macro-F1. This can reduce sampling bias and some text-duplicate leakage, but does not guarantee higher real-world accuracy.
- **Scores:** displayed percentages come from model output probabilities; they are not guaranteed calibrated probabilities of suitability. Top-three scores need not sum to 100%.
- **Evaluation:** do not compare role-level metrics with broad-category MGAT metrics as if they measured the same task.
- **PDF input:** text-based PDFs are supported. Scanned/image-only PDFs need OCR.
- **Responsible use:** use predictions as decision support, not as the sole basis for hiring. Audit performance across role groups and avoid inferring protected traits.
